# Reproduce Headline Statistics

This notebook reproduces the key statistics from the Urban Green and Blue Change Monitor without the web application. Run this in Google Colab or locally with GEE authentication.

## Prerequisites

```bash
pip install earthengine-api numpy pandas matplotlib
```

Authenticate with GEE:

```python
import ee
ee.Authenticate()  # Opens browser
ee.Initialize(project='graphic-tide-513321-m3')
```

In [ ]:
# Configuration
PROJECT_ID = 'graphic-tide-513321-m3'
STUDY_YEARS = list(range(2018, 2026))  # 2018-2025
SEASON = 'oct-dec'  # Oct 1 - Dec 31
SCALE = 20
CLOUD_THRESHOLD = 80

# District assets
DISTRICTS = {
    'bengaluru_urban': {
        'asset_id': 'projects/graphic-tide-513321-m3/assets/Bengaluru',
        'feature_index': 1,
        'name': 'Bengaluru Urban'
    },
    'gautam_buddha_nagar': {
        'asset_id': 'projects/graphic-tide-513321-m3/assets/GautamBuddhaNagar',
        'feature_index': 0,
        'name': 'Gautam Buddha Nagar'
    }
}

# Classification thresholds
THRESHOLDS = {
    'mndwi_water': 0.10,
    'ndvi_water_max': 0.15,
    'ndvi_veg_min': 0.35,
    'ndbi_built_min': 0.0,
    'ndvi_built_max': 0.20
}

In [ ]:
import ee
import numpy as np
import pandas as pd
import json
from datetime import datetime

# Initialize GEE
ee.Initialize(project=PROJECT_ID)
print('GEE initialized successfully')

In [ ]:
def get_district_geometry(district_key):
    """Get district geometry from GEE asset."""
    district = DISTRICTS[district_key]
    fc = ee.FeatureCollection(district['asset_id'])
    feature = fc.toList(fc.size()).get(district['feature_index'])
    return ee.Feature(feature).geometry()


def get_season_dates(year, season='oct-dec'):
    """Get start and end dates for season."""
    if season == 'oct-dec':
        return f'{year}-10-01', f'{year+1}-01-01'
    elif season == 'jan-mar':
        return f'{year}-01-01', f'{year}-04-01'
    elif season == 'mar-may':
        return f'{year}-03-01', f'{year}-06-01'
    elif season == 'jun-sep':
        return f'{year}-06-01', f'{year}-10-01'
    else:
        raise ValueError(f'Unknown season: {season}')


def mask_sentinel2_clouds(image):
    """Apply cloud mask using SCL band."""
    scl = image.select('SCL')
    mask = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
            .And(scl.neq(8)).And(scl.neq(9))
            .And(scl.neq(10)).And(scl.neq(11)))
    return image.updateMask(mask).select(['B2','B3','B4','B8','B11','B12'])\
        .multiply(0.0001).copyProperties(image, ['system:time_start', 'system:index_id', 'CLOUDY_PIXEL_PERCENTAGE'])


def compute_indices(image):
    """Compute spectral indices."""
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return image.addBands([ndvi, ndwi, mndwi, ndbi])


def classify_spectral(image, thresholds):
    """Classify using spectral rules."""
    ndvi = image.select('NDVI')
    mndwi = image.select('MNDWI')
    ndbi = image.select('NDBI')

    water = mndwi.gt(thresholds['mndwi_water']).And(ndvi.lt(thresholds['ndvi_water_max']))
    vegetation = ndvi.gte(thresholds['ndvi_veg_min']).And(water.Not())
    built_bare = ndbi.gt(thresholds['ndbi_built_min']).And(ndvi.lt(thresholds['ndvi_built_max'])).And(water.Not())
    valid_mask = ndvi.mask()
    other = valid_mask.And(water.Not()).And(vegetation.Not()).And(built_bare.Not())

    classification = (water.multiply(1)
                      .add(vegetation.multiply(2))
                      .add(built_bare.multiply(3))
                      .add(other.multiply(4))
                      .rename('landcover').byte())
    return image.addBands(classification)


def classify_dynamic_world(geometry, start_date, end_date):
    """Classify using Dynamic World."""
    dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
          .filterBounds(geometry)
          .filterDate(start_date, end_date)
          .select('label')
          .mode()
          .rename('landcover'))
    # Remap DW classes to our 4 classes
    remap = ee.Image.constant([
        [0, 1],   # water
        [1, 2],   # trees
        [2, 2],   # grass
        [3, 1],   # flooded vegetation
        [4, 2],   # crops
        [5, 2],   # shrub
        [6, 3],   # built
        [7, 3],   # bare
        [8, 4],   # snow
    ])
    return dw.remap(remap.slice(1, 0), remap.slice(1, 1), 4, 'landcover').byte()


def compute_areas(classification, geometry, scale, raw_area_km2):
    """Compute area statistics for each class."""
    pixel_area = ee.Image.pixelArea().divide(1e6)  # km²
    class_with_area = classification.addBands(pixel_area.rename('area_km2'))

    stats = class_with_area.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=0, groupName='class'),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True,
        crs='EPSG:32643'
    )

    valid_area = pixel_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True,
        crs='EPSG:32643'
    ).get('area_km2')

    groups = ee.List(stats.get('groups')).getInfo()
    valid_km2 = valid_area.getInfo() if valid_area else 0

    class_areas = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for group in groups:
        class_id = int(group['class'])
        area = float(group['sum'])
        class_areas[class_id] = area

    denom = valid_km2 if valid_km2 > 0 else 1
    return {
        'water_km2': class_areas[1],
        'vegetation_km2': class_areas[2],
        'built_bare_km2': class_areas[3],
        'other_km2': class_areas[4],
        'water_pct': class_areas[1] / denom * 100,
        'vegetation_pct': class_areas[2] / denom * 100,
        'built_bare_pct': class_areas[3] / denom * 100,
        'other_pct': class_areas[4] / denom * 100,
        'valid_km2': valid_km2,
        'raw_district_km2': raw_area_km2,
    }


def compute_mean_indices(composite, geometry, scale):
    """Compute mean indices."""
    means = composite.select(['NDVI', 'MNDWI', 'NDBI']).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True,
        crs='EPSG:32643'
    ).getInfo()
    return {
        'mean_ndvi': means.get('NDVI', 0.0),
        'mean_mndwi': means.get('MNDWI', 0.0),
        'mean_ndbi': means.get('NDBI', 0.0),
    }


def get_composite_dates(collection):
    """Get first and last image dates."""
    dates = collection.aggregate_array('system:time_start').getInfo()
    if not dates:
        return None, None
    dates.sort()
    first = datetime.fromtimestamp(dates[0] / 1000).strftime('%Y-%m-%d')
    last = datetime.fromtimestamp(dates[-1] / 1000).strftime('%Y-%m-%d')
    return first, last


def run_analysis_for_district(district_key, classifier='spectral'):
    """Run complete analysis for a district."""
    district = DISTRICTS[district_key]
    geometry = get_district_geometry(district_key)
    raw_area_km2 = geometry.area(1).divide(1e6).getInfo()

    print(f'\nAnalyzing {district["name"]} (raw area: {raw_area_km2:.2f} km²)')
    print(f'Classifier: {classifier}')
    print(f'Years: {STUDY_YEARS}')
    print(f'Season: {SEASON}')
    print(f'Scale: {SCALE}m')

    results = []
    for year in STUDY_YEARS:
        start_date, end_date = get_season_dates(year, SEASON)

        if classifier == 'spectral':
            collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                          .filterBounds(geometry)
                          .filterDate(start_date, end_date)
                          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
                          .map(mask_sentinel2_clouds))
            images_used = collection.size().getInfo()

            if images_used == 0:
                print(f'  {year}: NO IMAGES')
                results.append({'year': year, 'images_used': 0, 'valid_km2': 0})
                continue

            composite = collection.median().clip(geometry)
            composite = compute_indices(composite)
            composite = classify_spectral(composite, THRESHOLDS)
            first_date, last_date = get_composite_dates(collection)
        else:
            composite = classify_dynamic_world(geometry, start_date, end_date)
            images_used = 1
            first_date, last_date = start_date, end_date

        areas = compute_areas(composite.select('landcover'), geometry, SCALE, raw_area_km2)
        means = compute_mean_indices(composite, geometry, SCALE) if classifier == 'spectral' else {'mean_ndvi': 0, 'mean_mndwi': 0, 'mean_ndbi': 0}

        result = {
            'year': year,
            'images_used': images_used,
            'valid_km2': areas['valid_km2'],
            'vegetation_km2': areas['vegetation_km2'],
            'water_km2': areas['water_km2'],
            'built_bare_km2': areas['built_bare_km2'],
            'other_km2': areas['other_km2'],
            'vegetation_pct': areas['vegetation_pct'],
            'water_pct': areas['water_pct'],
            'built_bare_pct': areas['built_bare_pct'],
            'other_pct': areas['other_pct'],
            'mean_ndvi': means['mean_ndvi'],
            'mean_mndwi': means['mean_mndwi'],
            'mean_ndbi': means['mean_ndbi'],
            'composite_first_date': first_date,
            'composite_last_date': last_date,
        }
        results.append(result)
        print(f'  {year}: {images_used} images, veg={areas["vegetation_km2"]:.2f} km², water={areas["water_km2"]:.2f} km², built={areas["built_bare_km2"]:.2f} km²')

    return pd.DataFrame(results)

print('Functions defined successfully')

In [ ]:
# Run analysis for both districts
print('='*60)
print('RUNNING ANALYSIS')
print('='*60)

# Bengaluru Urban - Spectral
bengaluru_spectral = run_analysis_for_district('bengaluru_urban', 'spectral')

# Gautam Buddha Nagar - Spectral
gbn_spectral = run_analysis_for_district('gautam_buddha_nagar', 'spectral')

# Bengaluru Urban - Dynamic World
bengaluru_dw = run_analysis_for_district('bengaluru_urban', 'dynamic_world')

# Gautam Buddha Nagar - Dynamic World
gbn_dw = run_analysis_for_district('gautam_buddha_nagar', 'dynamic_world')

print('\nAnalysis complete!')

In [ ]:
# Display results
print('\n' + '='*60)
print('BENGALURU URBAN - SPECTRAL')
print('='*60)
print(bengaluru_spectral[['year', 'images_used', 'valid_km2',
                          'vegetation_km2', 'water_km2', 'built_bare_km2',
                          'vegetation_pct', 'water_pct', 'built_bare_pct',
                          'mean_ndvi', 'mean_mndwi', 'mean_ndbi']].to_string(index=False))

In [ ]:
print('\n' + '='*60)
print('GAUTAM BUDDHA NAGAR - SPECTRAL')
print('='*60)
print(gbn_spectral[['year', 'images_used', 'valid_km2',
                    'vegetation_km2', 'water_km2', 'built_bare_km2',
                    'vegetation_pct', 'water_pct', 'built_bare_pct',
                    'mean_ndvi', 'mean_mndwi', 'mean_ndbi']].to_string(index=False))

In [ ]:
print('\n' + '='*60)
print('BENGALURU URBAN - DYNAMIC WORLD')
print('='*60)
print(bengaluru_dw[['year', 'images_used', 'valid_km2',
                    'vegetation_km2', 'water_km2', 'built_bare_km2',
                    'vegetation_pct', 'water_pct', 'built_bare_pct']].to_string(index=False))

In [ ]:
print('\n' + '='*60)
print('GAUTAM BUDDHA NAGAR - DYNAMIC WORLD')
print('='*60)
print(gbn_dw[['year', 'images_used', 'valid_km2',
              'vegetation_km2', 'water_km2', 'built_bare_km2',
              'vegetation_pct', 'water_pct', 'built_bare_pct']].to_string(index=False))

In [ ]:
# Key headline statistics: 2018 vs 2025 change
print('\n' + '='*60)
print('HEADLINE CHANGES (2018 → 2025)')
print('='*60)

for name, df_spectral, df_dw in [
    ('Bengaluru Urban', bengaluru_spectral, bengaluru_dw),
    ('Gautam Buddha Nagar', gbn_spectral, gbn_dw),
]:
    print(f'\n{name}:')
    
    # Get 2018 and 2025 stats
    y2018_spec = df_spectral[df_spectral['year'] == 2018].iloc[0] if 2018 in df_spectral['year'].values else None
    y2025_spec = df_spectral[df_spectral['year'] == 2025].iloc[0] if 2025 in df_spectral['year'].values else None
    y2018_dw = df_dw[df_dw['year'] == 2018].iloc[0] if 2018 in df_dw['year'].values else None
    y2025_dw = df_dw[df_dw['year'] == 2025].iloc[0] if 2025 in df_dw['year'].values else None

    if y2018_spec is not None and y2025_spec is not None:
        print('  Spectral Classifier:')
        for cls, label in [('vegetation', 'Vegetation'), ('water', 'Water'), ('built_bare', 'Built-up/Bare')]:
            km2_2018 = y2018_spec[f'{cls}_km2']
            km2_2025 = y2025_spec[f'{cls}_km2']
            pct_2018 = y2018_spec[f'{cls}_pct']
            pct_2025 = y2025_spec[f'{cls}_pct']
            change_km2 = km2_2025 - km2_2018
            change_pct = (change_km2 / km2_2018 * 100) if km2_2018 > 0 else 0
            direction = 'gain' if change_km2 > 0 else 'loss'
            print(f'    {label}: {km2_2018:.2f} → {km2_2025:.2f} km² ({pct_2018:.1f}% → {pct_2025:.1f}%) | {direction} {abs(change_km2):.2f} km² ({abs(change_pct):.1f}%)')
        
        # NDVI change
        ndvi_change = y2025_spec['mean_ndvi'] - y2018_spec['mean_ndvi']
        print(f'    Mean NDVI: {y2018_spec["mean_ndvi"]:.3f} → {y2025_spec["mean_ndvi"]:.3f} | change: {ndvi_change:+.3f}')

    if y2018_dw is not None and y2025_dw is not None:
        print('  Dynamic World Classifier:')
        for cls, label in [('vegetation', 'Vegetation'), ('water', 'Water'), ('built_bare', 'Built-up/Bare')]:
            km2_2018 = y2018_dw[f'{cls}_km2']
            km2_2025 = y2025_dw[f'{cls}_km2']
            pct_2018 = y2018_dw[f'{cls}_pct']
            pct_2025 = y2025_dw[f'{cls}_pct']
            change_km2 = km2_2025 - km2_2018
            change_pct = (change_km2 / km2_2018 * 100) if km2_2018 > 0 else 0
            direction = 'gain' if change_km2 > 0 else 'loss'
            print(f'    {label}: {km2_2018:.2f} → {km2_2025:.2f} km² ({pct_2018:.1f}% → {pct_2025:.1f}%) | {direction} {abs(change_km2):.2f} km² ({abs(change_pct):.1f}%)')

In [ ]:
# Save results to CSV for reference
bengaluru_spectral.to_csv('bengaluru_urban_spectral.csv', index=False)
gbn_spectral.to_csv('gautam_buddha_nagar_spectral.csv', index=False)
bengaluru_dw.to_csv('bengaluru_urban_dynamic_world.csv', index=False)
gbn_dw.to_csv('gautam_buddha_nagar_dynamic_world.csv', index=False)

print('Results saved to CSV files')

## Trend Analysis (Mann-Kendall, Sen's Slope)

In [ ]:
def mann_kendall_test(values):
    """Mann-Kendall trend test."""
    n = len(values)
    if n < 3:
        return {'tau': 0, 'p_value': 1, 's': 0, 'trend': 'insufficient_data'}
    
    s = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            if values[j] > values[i]:
                s += 1
            elif values[j] < values[i]:
                s -= 1
    
    # Tie correction
    unique = {}
    for v in values:
        unique[v] = unique.get(v, 0) + 1
    tie_corr = sum(c * (c - 1) * (2 * c + 5) for c in unique.values() if c > 1)
    var_s = (n * (n - 1) * (2 * n + 5) - tie_corr) / 18
    
    if var_s == 0:
        return {'tau': 0, 'p_value': 1, 's': s, 'trend': 'no_trend'}
    
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0
    
    from scipy import stats
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    tau = s / (n * (n - 1) / 2)
    
    if p < 0.05:
        trend = 'increasing' if s > 0 else 'decreasing'
    else:
        trend = 'no trend'
    
    return {'tau': tau, 'p_value': p, 's': s, 'trend': trend, 'significant': p < 0.05}


def sen_slope(values, years, confidence=0.95):
    """Sen's slope estimator."""
    n = len(values)
    if n < 2:
        return {'slope': 0, 'ci_lower': 0, 'ci_upper': 0}
    
    slopes = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            if years[j] != years[i]:
                slopes.append((values[j] - values[i]) / (years[j] - years[i]))
    
    if not slopes:
        return {'slope': 0, 'ci_lower': 0, 'ci_upper': 0}
    
    slopes.sort()
    median_slope = slopes[len(slopes) // 2]
    
    # Confidence interval
    from scipy import stats
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha / 2)
    C = n * (n - 1) / 2
    Ru = int(C / 2 - z * np.sqrt(C * (2 * n + 5) / 18))
    Rl = int(C / 2 + z * np.sqrt(C * (2 * n + 5) / 18))
    Ru = max(0, min(Ru, len(slopes) - 1))
    Rl = max(0, min(Rl, len(slopes) - 1))
    
    return {'slope': median_slope, 'ci_lower': slopes[Ru], 'ci_upper': slopes[Rl]}

print('Trend functions defined')

In [ ]:
# Run trend analysis for each district and variable
print('\n' + '='*60)
print('TREND ANALYSIS (2018-2025)')
print('='*60)
print('WARNING: n=8 gives very low statistical power!')
print('='*60)

for name, df in [('Bengaluru Urban (Spectral)', bengaluru_spectral),
                  ('Gautam Buddha Nagar (Spectral)', gbn_spectral),
                  ('Bengaluru Urban (DW)', bengaluru_dw),
                  ('Gautam Buddha Nagar (DW)', gbn_dw)]:
    print(f'\n{name}:')
    
    for var, label in [('vegetation_pct', 'Vegetation %'),
                        ('water_pct', 'Water %'),
                        ('built_bare_pct', 'Built-up %'),
                        ('mean_ndvi', 'Mean NDVI')]:
        if var not in df.columns:
            continue
        values = df[var].tolist()
        years = df['year'].tolist()
        
        mk = mann_kendall_test(values)
        sen = sen_slope(values, years)
        
        sig = '*' if mk['significant'] else ''
        print(f'  {label}: τ={mk["tau"]:.3f}, p={mk["p_value"]:.4f} {sig}, Sen slope={sen["slope"]:.4f}/yr (95% CI: {sen["ci_lower"]:.4f} to {sen["ci_upper"]:.4f})')